# Notebook 3: finding a replacement

Give it a player, get back the players who would replace him best.

"Best" here means **plays the same way**, not "scores the same amount". Two
strikers with ten goals each can be completely different players: one lives in
the box and wins headers, the other drifts wide, takes shots from distance and
gets fouled a lot. A club replacing the first one with the second has not
replaced anything.

So the comparison is built on style: shots and where they come from, crosses,
interceptions, tackles won, aerial duels, fouls drawn, offsides, recoveries. All
per 90 minutes, all compared against other players in the same position.

### The hard part is proving it works

There is no labelled answer to "who replaces this player". Nobody publishes a
list of correct replacements to test against.

But there is a clean stand-in. A player's own next season is the best possible
replacement for his current one: same person, same habits, one year older. If
the features really capture a personal style, then searching with a player's
2023/24 profile should find his own 2024/25 profile at or near the top, out of
every player in his position that season.

That turns a fuzzy question into numbers. We report how often the real player
comes back first (hit@1), in the top 5, and in the top 10, and compare methods
on it, including a neural autoencoder.

### Before you run this

Run it with `jupyter notebook` from the terminal, not inside VS Code. Smart App
Control blocks some compiled scikit-learn files under the VS Code kernel.

Notebook 2 should have been run once, since the budget filter uses the market
values it saves. Without them everything still works except that filter.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.decomposition import PCA

from src.config import PROCESSED_DIR, MODELS_DIR
from src import similarity as S
from src.market import euros

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
plt.rcParams["figure.figsize"] = (10, 4)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
print("ready")

## 1. Player profiles

Everything comes from the FBref tables collected by `scripts/04_fetch_players.py`:
standard, shooting, misc, playing time and keeper. They all come from the same
source, so they join to each other exactly on player, club and season. No fuzzy
name matching anywhere in the profiles.

Two details handled on the way in:

**Mid-season transfers.** A player who moved in January has one row per club.
Those get merged into a single season, counting stats summed and rates weighted
by minutes, with the club he played most for kept as his team.

**Column names.** FBref names change between scraper versions, and many stats
appear twice, once as a total and once per 90. The module finds each stat by its
shape and refuses to take a per 90 or percentage column when it wants a total.
Anything it cannot find is listed below rather than silently dropped.

In [ ]:
profiles, found, missing = S.build_profiles()

print(f"{profiles['player_key'].nunique():,} players, "
      f"seasons {profiles['season_start'].min()} to {profiles['season_start'].max()}\n")
print(f"stats found   ({len(found)}): {', '.join(found)}")
print(f"stats missing ({len(missing)}): {', '.join(missing) if missing else 'none'}")

outfield_found = [f for f in found if f not in S.KEEPER_NAMES]
if len(outfield_found) < 8:
    print("\nWARNING: fewer than 8 outfield stats found. Similarity will be weak.")
    print("Print the headers of data/interim/players_misc.csv and check the patterns in")
    print("OUTFIELD_FEATURES in src/similarity.py.")

In [ ]:
summary = profiles.groupby("pos_group").agg(
    player_seasons=("player", "count"),
    regulars=("minutes", lambda m: int((m >= S.MIN_MINUTES).sum())),
    median_minutes=("minutes", "median"),
).reindex(list(S.POSITION_GROUPS))
summary

## 2. Small samples lie

Per 90 numbers are unreliable for anyone who barely played. A striker who came
on twice and scored once has a goals per 90 figure that would embarrass Haaland,
and it means nothing.

Two things deal with that.

**A minutes threshold.** Only players with at least 900 minutes, ten full
matches, are used as queries or offered as replacements.

**Shrinkage.** Every per 90 figure gets pulled toward the average for that
position, and the pull is strongest for players with few minutes. Technically it
adds five matches of perfectly average output to everyone's record. A regular
with 3,000 minutes barely moves. A player with 200 minutes gets dragged most of
the way back to normal, which is the honest thing to believe about him.

The table below shows the effect on the forwards with the highest raw rates
from limited minutes.

In [ ]:
if "goals_p90_raw" in profiles.columns:
    small = profiles[(profiles["pos_group"] == "FW") & profiles["minutes"].between(90, 600)]
    demo = small.nlargest(8, "goals_p90_raw")[
        ["player", "team", "season_start", "minutes", "goals", "goals_p90_raw", "goals_p90"]
    ].rename(columns={"goals_p90_raw": "raw per 90", "goals_p90": "shrunk per 90"})
    print("The raw figures are fantasy. The shrunk ones are believable.\n")
    print(demo.round(2).to_string(index=False))

## 3. How we measure it

The test described at the top. For each of the last three pairs of seasons, take
every regular in a position, search with his profile from the first season among
all regulars in that position in the second season, and record where his own
profile ranks.

A player who changed position between seasons is skipped, since he is no longer
in the pool he is searched against.

The `random_hit@10` column is what pure chance would score, so every other
number has something to be compared against.

Four spaces get tested:

**Goals and assists only.** The naive version, and the one most people would
build first. It is here to show why it is not enough.

**Full style, raw per 90.** Every stat, no shrinkage.

**Full style, shrunk per 90.** Every stat, with the shrinkage above.

**PCA and an autoencoder.** Both compress the style stats into a handful of
dimensions. PCA does it with straight lines, the autoencoder with a small neural
network. Both are fitted only on seasons before the evaluation seasons.

Every stat is standardised within its position before comparing, so a centre
back's interceptions are measured against other centre backs, not against
strikers.

In [ ]:
seasons = sorted(int(s) for s in profiles["season_start"].unique())
LAST = int(seasons[-1])
EVAL_PAIRS = [(s, s + 1) for s in range(LAST - 3, LAST)]
TRAIN_SEASONS = [s for s in seasons if s < LAST - 3]

print("evaluation pairs:", EVAL_PAIRS)
print("seasons used to fit PCA and the autoencoder:", TRAIN_SEASONS)

baseline_cols = [c for c in ("goals_p90", "assists_p90") if c in profiles.columns]

spaces = {
    "goals and assists only": S.style_space(profiles, columns=baseline_cols),
    "full style, raw per 90": S.style_space(profiles, raw=True),
    "full style, shrunk per 90": S.style_space(profiles),
}

def evaluate(space):
    return S.self_consistency(profiles, space, EVAL_PAIRS)

results = {name: evaluate(space) for name, space in spaces.items()}
print(f"\n{results['full style, shrunk per 90']['queries']:,} test searches per method")

### PCA

Many of these stats move together. Shots and shots on target, tackles and
recoveries. PCA finds the few directions that carry most of the variation and
throws away the rest, which can remove noise, or can throw away exactly the
small differences that make a player recognisable.

In [ ]:
z_style = spaces["full style, shrunk per 90"]
o_cols = [c for c in z_style.columns if c.startswith("o_")]
k_cols = [c for c in z_style.columns if c.startswith("k_")]

outfield = profiles["pos_group"].isin(S.OUTFIELD_GROUPS)
fit_rows = (outfield & profiles["season_start"].isin(TRAIN_SEASONS)
            & (profiles["minutes"] >= S.MIN_MINUTES))

pca = PCA(n_components=0.9, random_state=SEED).fit(z_style.loc[fit_rows, o_cols])
print(f"PCA keeps {pca.n_components_} of {len(o_cols)} dimensions for 90% of the variance")

def embed_space(values, prefix):
    emb = pd.DataFrame(0.0, index=profiles.index,
                       columns=[f"{prefix}{i}" for i in range(values.shape[1])])
    emb.loc[outfield] = values
    # Keepers keep their own stats, the compression is for outfield players only
    return pd.concat([emb, z_style[k_cols]], axis=1)

spaces["PCA"] = embed_space(pca.transform(z_style.loc[outfield, o_cols]), "pca_")
results["PCA"] = evaluate(spaces["PCA"])

### Autoencoder

A small network that squeezes each profile down to six numbers and then tries to
rebuild the original from them. If it can rebuild it well, those six numbers
hold the essence of how the player plays. Unlike PCA it can learn curved
relationships between stats.

Early stopping on a held-out slice of the training seasons, same as the other
notebooks.

In [ ]:
class StyleAutoencoder(nn.Module):
    def __init__(self, n_features, latent=6):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(n_features, 16), nn.ReLU(), nn.Linear(16, latent))
        self.decoder = nn.Sequential(nn.Linear(latent, 16), nn.ReLU(), nn.Linear(16, n_features))

    def forward(self, x):
        return self.decoder(self.encoder(x))


X_fit = torch.tensor(z_style.loc[fit_rows, o_cols].to_numpy(), dtype=torch.float32)
order = torch.randperm(len(X_fit))
n_val = max(1, len(X_fit) // 10)
X_val, X_tr = X_fit[order[:n_val]], X_fit[order[n_val:]]

LATENT = min(6, len(o_cols) - 1)
ae = StyleAutoencoder(len(o_cols), LATENT)
opt = torch.optim.AdamW(ae.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.MSELoss()
loader = torch.utils.data.DataLoader(X_tr, batch_size=64, shuffle=True)

best, best_state, wait, history = np.inf, None, 0, []
for epoch in range(500):
    ae.train()
    for xb in loader:
        opt.zero_grad()
        loss_fn(ae(xb), xb).backward()
        opt.step()
    ae.eval()
    with torch.no_grad():
        train_loss = loss_fn(ae(X_tr), X_tr).item()
        val_loss = loss_fn(ae(X_val), X_val).item()
    history.append((train_loss, val_loss))
    if val_loss < best - 1e-5:
        best, wait = val_loss, 0
        best_state = {k: v.clone() for k, v in ae.state_dict().items()}
    else:
        wait += 1
        if wait >= 40:
            break

ae.load_state_dict(best_state)
ae.eval()
print(f"stopped after {len(history)} epochs, reconstruction error {best:.4f}")
print("(1.0 would mean it learned nothing, since the inputs are standardised)")

with torch.no_grad():
    latent = ae.encoder(torch.tensor(z_style.loc[outfield, o_cols].to_numpy(),
                                     dtype=torch.float32)).numpy()

spaces["autoencoder"] = embed_space(latent, "ae_")
results["autoencoder"] = evaluate(spaces["autoencoder"])

h = np.array(history)
plt.plot(h[:, 0], label="train"); plt.plot(h[:, 1], label="validation")
plt.xlabel("epoch"); plt.ylabel("reconstruction error"); plt.title("Autoencoder training")
plt.legend(); plt.tight_layout(); plt.show()

## 4. Results

Read `hit@5` as: out of every regular who played two seasons in a row in the
same position, how often did his own next season land in the top five matches.
Then look at `random_hit@10` to remember what luck alone would score.

In [ ]:
table = pd.DataFrame(results).T
for c in ("hit@1", "hit@5", "hit@10", "random_hit@10"):
    table[c] = (table[c] * 100).round(1)
table["median_rank"] = table["median_rank"].round(0)
table["queries"] = table["queries"].astype(int)
table

In [ ]:
ax = table["hit@5"].sort_values().plot(kind="barh", color="#1565c0")
ax.axvline(table["random_hit@10"].mean() / 2, color="#c62828", ls="--",
           label="roughly what chance scores at top 5")
ax.set_xlabel("own next season found in the top 5 (%)")
ax.set_title("Which space recognises a player best")
ax.legend(); plt.tight_layout(); plt.show()

### Picking the space

The shrunk style space has one big advantage over PCA and the autoencoder: every
dimension is a real football stat, so a match can be explained. "Similar because
both win lots of aerials and rarely shoot from distance" is something a scout can
act on. Six autoencoder numbers are not.

So the rule is: use the interpretable space unless a compressed one beats it by a
clear margin, at least two points of hit@5. Anything smaller than that is inside
the noise of one season's evaluation.

In [ ]:
INTERPRETABLE = "full style, shrunk per 90"
BEST = table["hit@5"].idxmax()
margin = table.loc[BEST, "hit@5"] - table.loc[INTERPRETABLE, "hit@5"]

if BEST != INTERPRETABLE and margin < 2:
    print(f"{BEST} scored highest, but only by {margin:.1f} points.")
    BEST = INTERPRETABLE

space = spaces[BEST]
print(f"Using: {BEST}")

## 5. Market values

Joined from notebook 2 so replacements can be filtered by budget. Transfermarkt
and FBref spell names differently, so this matches on name and season first and
then on surname plus club for whatever is left.

Anyone who cannot be matched has no value, and gets dropped whenever a budget is
set. The match rate below tells you how many that is.

In [ ]:
mv_path = PROCESSED_DIR / "player_market_values.csv"
HAS_VALUES = mv_path.exists()

if HAS_VALUES:
    profiles = S.attach_market_value(profiles, pd.read_csv(mv_path))
    latest = profiles[(profiles["season_start"] == LAST) & (profiles["minutes"] >= S.MIN_MINUTES)]
    print(f"market value found for {latest['market_value_eur'].notna().mean():.1%} "
          f"of {LAST}/{(LAST + 1) % 100:02d} regulars")
else:
    print("No player_market_values.csv. Run notebook 2 to enable budget filters.")

## 6. Finding a replacement

Set `PLAYER` to anyone. Leave it as `None` and the notebook picks the top
scoring regular forward from the latest season.

Replacements are drawn from the latest season only, since those are the players
who actually exist to be bought. The player's own club is excluded, and he is
only compared against his own position group.

In [ ]:
PLAYER = None

if PLAYER is None:
    pool = profiles[(profiles["season_start"] == LAST) & (profiles["pos_group"] == "FW")
                    & (profiles["minutes"] >= S.MIN_MINUTES)]
    PLAYER = pool.nlargest(1, "goals")["player"].iloc[0]

query = S.resolve_player(profiles, PLAYER)
row = profiles.loc[query]
age = int(row["age"]) if pd.notna(row["age"]) else "?"
print(f"{row['player']}, {row['team']}, {row['pos_group']}, "
      f"{row['season_start']}/{(row['season_start'] + 1) % 100:02d}, "
      f"{int(row['minutes'])} minutes, age {age}")


def show(frame):
    out = frame.copy()
    out["similarity"] = out["similarity"].round(3)
    out["age"] = out["age"].round(0)
    if "market_value_eur" in out:
        out["market_value_eur"] = out["market_value_eur"].map(
            lambda v: euros(v) if pd.notna(v) else "unknown")
    return out


matches = S.find_replacements(profiles, space, PLAYER, n=10)
show(matches)

In [ ]:
if HAS_VALUES:
    query_value = profiles.loc[query, "market_value_eur"]
    BUDGET = query_value * 0.5 if pd.notna(query_value) else 20_000_000
    MAX_AGE = 25

    cheaper = S.find_replacements(profiles, space, PLAYER, n=10,
                                  max_age=MAX_AGE, max_value=BUDGET)
    print(f"Aged {MAX_AGE} or under, valued at {euros(BUDGET)} or less\n")
    print(show(cheaper).to_string())
    if len(cheaper) < 3:
        print("\nFew players pass both filters. Raise BUDGET or MAX_AGE to widen the search.")
else:
    print("Budget filter needs notebook 2's market values.")

## 7. Why they match

A similarity score on its own asks you to take the model's word for it. This
breaks the top match down stat by stat, as percentiles within the position. A
percentile of 90 means better than 90 percent of regulars in that position.

The top of the table is where the two players are most alike, the bottom is
where they differ most. Both halves matter to a scout.

In [ ]:
if len(matches):
    top = matches.index[0]
    comparison = S.compare_players(profiles, query, top)
    print("Most alike:")
    print(comparison.head(6).to_string(index=False))
    print("\nMost different:")
    print(comparison.tail(4).to_string(index=False))

In [ ]:
def radar(a, b):
    group = profiles.loc[a, "pos_group"]
    cols = [c for c in S.RADAR.get(group, []) if c in profiles.columns]
    if len(cols) < 3:
        print("Not enough stats for a radar chart in this position.")
        return
    pct = S.percentiles(profiles, cols)
    angles = np.linspace(0, 2 * np.pi, len(cols), endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={"polar": True})
    for idx, colour in ((a, "#1565c0"), (b, "#c62828")):
        values = pct.loc[idx, cols].fillna(0).tolist()
        values += values[:1]
        ax.plot(angles, values, color=colour, linewidth=2, label=profiles.loc[idx, "player"])
        ax.fill(angles, values, color=colour, alpha=0.15)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([S.label(c) for c in cols], fontsize=9)
    ax.set_ylim(0, 100)
    ax.set_title("Percentile within position", pad=20)
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout(); plt.show()


if len(matches):
    radar(query, matches.index[0])

## 8. Save for the dashboard

The profiles and the chosen space are saved together, so the dashboard can load
them and search instantly without rebuilding anything or loading a neural
network. The evaluation results go into the metadata so the dashboard can show
how trustworthy the matches are.

In [ ]:
path = S.save_profiles(profiles, space)

meta = {
    "method": BEST,
    "min_minutes": S.MIN_MINUTES,
    "prior_nineties": S.PRIOR_NINETIES,
    "latest_season": LAST,
    "outfield_features": S.feature_columns(profiles, "FW"),
    "keeper_features": S.feature_columns(profiles, "GK"),
    "evaluation": table.reset_index().rename(columns={"index": "space"}).to_dict(orient="records"),
}
(MODELS_DIR / "similarity_meta.json").write_text(json.dumps(meta, indent=2, default=float))
print(f"saved {path.name} and similarity_meta.json")

reloaded, reloaded_space = S.load_profiles()
check = S.find_replacements(reloaded, reloaded_space, PLAYER, n=3)
print("reloaded from disk, same top match:",
      len(check) > 0 and check["player"].iloc[0] == matches["player"].iloc[0])

## Where this leaves us

A replacement finder with an honest measure of how well it works, and a saved
index the dashboard can search in milliseconds.

### Limitations worth writing up

**Style is only as rich as the stats.** Your FBref scrape only offered basic stat
types, so there are no progressive passes, carries, touches in the box or
pressures. Those are exactly the stats that separate a deep playmaker from a
box-to-box midfielder, and midfielders are where you should expect the weakest
matches.

**Replacement is not only style.** The model knows nothing about injuries,
contract situations, wages, whether a player would move, or how he fits a
particular manager's system. It narrows a list of hundreds down to ten worth
watching. It does not replace watching them.

**The self-consistency test rewards stable players.** Someone whose role changed,
say a winger converted to a wing back, will score as a miss even though the model
may be right that he now plays differently. Some of the misses are the player
changing, not the model failing.

**Positions are coarse.** Four groups put a full back and a centre back in the
same pool. The position filter keeps the search sensible, but finer positions
would make it sharper.

**Market value matching is imperfect.** Players the join could not match simply
vanish from budget filtered searches. Check the match rate above before trusting
a filtered list.

### Next

The dashboard. All three notebooks now save what it needs:

`models/match_net.pt` and `data/processed/matches_features.csv` for match
prediction, `models/value_*` and `data/processed/player_market_values.csv` for
valuations, and `data/processed/player_profiles.csv` for replacements.